In [33]:
import cv2
import numpy as np
import os
import random

# Настройки
SOURCE_DIR = "/home/alterpath/drones/catkin_ws/src/hack_drones_20-22/augmented_dataset/source/"
OUTPUT_DIR = "/home/alterpath/drones/catkin_ws/src/hack_drones_20-22/augmented_dataset/train_images/"
FILES = ["1.jpg", "2.jpg", "3.jpg", "4.jpg", "5.jpg", "6.jpg", "7.jpg"] 
COUNT_PER_IMAGE = 150 
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

def create_variant(img, filename, index):
    h, w, _ = img.shape
    
    bg_color = random.randint(50, 180)
    background = np.full((h, w, 3), bg_color, dtype=np.uint8)
    noise = np.random.randint(-20, 20, (h, w, 3), dtype=np.int16)
    background = np.clip(background.astype(np.int16) + noise, 0, 255).astype(np.uint8)
    lower_white = np.array([220, 220, 220])
    upper_white = np.array([255, 255, 255])
    mask = cv2.inRange(img, lower_white, upper_white)

    object_mask = cv2.bitwise_not(mask)
    final_img = background.copy()
    final_img[object_mask > 0] = img[object_mask > 0]
    brightness = random.uniform(0.5, 1.3)
    final_img = cv2.convertScaleAbs(final_img, alpha=brightness, beta=0)
    save_name = f"aug_{index}_{filename}"
    cv2.imwrite(os.path.join(OUTPUT_DIR, save_name), final_img)

print("Начинаю генерацию датасета...")
for fname in FILES:
    path = os.path.join(SOURCE_DIR, fname)
    image = cv2.imread(path)
    if image is None:
        print(f"Ошибка: не найден файл {path}")
        continue
    
    for i in range(COUNT_PER_IMAGE):
        create_variant(image, fname, i)
    print(f"Готово для {fname}")

print(f"\nУспех! Датасет из {len(FILES) * COUNT_PER_IMAGE} фото создан в {OUTPUT_DIR}")

Начинаю генерацию датасета...
Готово для 5.jpg
Готово для 6.jpg
Готово для 7.jpg

Успех! Датасет из 300 фото создан в /home/alterpath/drones/catkin_ws/src/hack_drones_20-22/augmented_dataset/train_images/


In [77]:
import cv2
import numpy as np
import os
import random

# --- НАСТРОЙКИ ПУТЕЙ ---
INPUT_DIR = "/home/alterpath/drones/catkin_ws/src/hack_drones_20-22/augmented_dataset/source"
OUTPUT_DIR = "/home/alterpath/drones/catkin_ws/src/hack_drones_20-22/augmented_dataset/classifier_data"

# ОПРЕДЕЛЯЕМ КЛАССЫ
classes = {
    'background': ['0', '10', '11', '12', '13'],
    'cheburashka_mandarin': ['1', '2', '3', '4', '17', '18', '19', '20', '21'],
    'orange': ['5', '6', '7', '22', '23', '24', '25'],
    'cheburashka_only': ['14', '15', '16']
}

# Создаем папки
for cls in classes.keys():
    os.makedirs(os.path.join(OUTPUT_DIR, 'train', cls), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_DIR, 'val', cls), exist_ok=True)

print(f"[*] Начало обработки. Ищу файлы в: {INPUT_DIR}")

count = 0
# Проходим по всем файлам
for file in os.listdir(INPUT_DIR):
    # Проверяем расширения (и jpg, и jpeg)
    if not file.lower().endswith((".jpg", ".jpeg")):
        continue

    full_path = os.path.join(INPUT_DIR, file)
    img = cv2.imread(full_path)
    
    if img is None:
        print(f"[-] Ошибка чтения: {file}")
        continue

    # Определяем имя файла без расширения (номер)
    name_no_ext = os.path.splitext(file)[0]
    
    # Ищем, к какому классу относится этот номер
    target_folder = None
    for cls_name, numbers in classes.items():
        if name_no_ext in numbers:
            target_folder = cls_name
            break
    
    if target_folder is None:
        print(f"[!] Файл {file} не в списке классов, пропускаю.")
        continue

    print(f"[+] Размножаю {file} -> класс {target_folder}...")

    # Делаем 100 копий с аугментацией
    for i in range(100):
        # 1. Случайная яркость (сильнее разброс для устойчивости)
        alpha = random.uniform(0.6, 1.4)
        aug_img = cv2.convertScaleAbs(img, alpha=alpha, beta=random.randint(-20, 20))
        
        # 2. Шум
        noise = np.random.randint(0, 25, img.shape, dtype='uint8')
        aug_img = cv2.add(aug_img, noise)
        
        # 3. Случайный поворот на 90/180/270 градусов (полезно для дрона!)
        rotate_code = random.choice([None, cv2.ROTATE_90_CLOCKWISE, cv2.ROTATE_180, cv2.ROTATE_90_COUNTERCLOCKWISE])
        if rotate_code is not None:
            aug_img = cv2.rotate(aug_img, rotate_code)

        # 4. Распределение Train (80%) / Val (20%)
        subfolder = 'train' if i < 80 else 'val'
        
        save_name = f"aug_{i}_{file}"
        save_path = os.path.join(OUTPUT_DIR, subfolder, target_folder, save_name)
        cv2.imwrite(save_path, aug_img)
    
    count += 1

print(f"\n--- ГОТОВО ---")
print(f"Обработано исходников: {count}")
print(f"Всего создано картинок: {count * 100}")
print(f"Путь к датасету: {OUTPUT_DIR}")

[*] Начало обработки. Ищу файлы в: /home/alterpath/drones/catkin_ws/src/hack_drones_20-22/augmented_dataset/source
[+] Размножаю 1.jpg -> класс cheburashka_mandarin...
[+] Размножаю 2.jpg -> класс cheburashka_mandarin...
[+] Размножаю 17.jpeg -> класс cheburashka_mandarin...
[+] Размножаю 22.jpeg -> класс orange...
[+] Размножаю 13.jpeg -> класс background...
[+] Размножаю 19.jpeg -> класс cheburashka_mandarin...
[+] Размножаю 18.jpeg -> класс cheburashka_mandarin...
[+] Размножаю 12.jpeg -> класс background...
[+] Размножаю 23.jpeg -> класс orange...
[+] Размножаю 6.jpg -> класс orange...
[+] Размножаю 15.jpeg -> класс cheburashka_only...
[+] Размножаю 0.jpeg -> класс background...
[+] Размножаю 10.jpeg -> класс background...
[+] Размножаю 14.jpeg -> класс cheburashka_only...
[+] Размножаю 4.jpg -> класс cheburashka_mandarin...
[+] Размножаю 11.jpeg -> класс background...
[+] Размножаю 16.jpeg -> класс cheburashka_only...
[+] Размножаю 21.jpeg -> класс cheburashka_mandarin...
[+] Разм

In [9]:
import cv2
import numpy as np
import os
import random

# ПУТИ - ПРОВЕРЕНЫ ТОБОЙ
INPUT_DIR = "/home/alterpath/drones/catkin_ws/src/hack_drones_20-22/augmented_dataset/source/cheburashka_only"
OUTPUT_DIR = "/home/alterpath/drones/catkin_ws/src/hack_drones_20-22/augmented_dataset/classifier_data"

# Создаем папки для классификации
for cls in ['background',
            'cheburashka_mandarin',
            'orange',
            'cheburashka_only']:
    os.makedirs(os.path.join(OUTPUT_DIR, 'train', cls), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_DIR, 'val', cls), exist_ok=True)

print(f"[*] Ищу файлы в: {INPUT_DIR}")

count = 0
# Тот самый цикл, который у тебя работал
for root, dirs, files in os.walk(INPUT_DIR):
    for file in files:
        if file.lower().endswith(".jpg") or file.lower().endswith(".jpeg"):
            full_path = os.path.join(root, file)
            img = cv2.imread(full_path)
            
            if img is None:
                print(f"[-] Не удалось прочитать: {file}")
                continue

            # ОПРЕДЕЛЯЕМ КЛАСС (1-4 это Чебурашка, 5-7 это Апельсин)
            # Берем первую букву имени файла
            name_start = file[0] 
            folder = 'cheburashka_only'

            print(f"[+] Размножаю {file} в папку {folder}...")

            # Делаем 100 копий с легкими изменениями
            for i in range(80):
                # 1. Случайная яркость
                alpha = random.uniform(0.7, 1.3)
                aug_img = cv2.convertScaleAbs(img, alpha=alpha, beta=0)
                
                # 2. Шум (чтобы не было идеально белого фона)
                noise = np.random.randint(0, 30, img.shape, dtype='uint8')
                aug_img = cv2.add(aug_img, noise)

                # 3. Решаем: в train (80 шт) или в val (20 шт)
                subfolder = 'train' if i < 80 else 'val'
                
                save_path = os.path.join(OUTPUT_DIR, subfolder, folder, f"{i}_{file}")
                cv2.imwrite(save_path, aug_img)
            
            count += 1

print(f"\n--- ИТОГ ---")
print(f"Обработано исходников: {count}")
print(f"Данные готовы в: {OUTPUT_DIR}")

[*] Ищу файлы в: /home/alterpath/drones/catkin_ws/src/hack_drones_20-22/augmented_dataset/source/cheburashka_only
[+] Размножаю 144406bc-110f-4028-9f52-65054696eada.jpeg в папку cheburashka_only...
[+] Размножаю f128c761-ac8b-4e0e-a2da-b7731578ae5a.jpeg в папку cheburashka_only...
[+] Размножаю fc2e29ae-ac43-42c8-ac75-13ae76eb84a5.jpeg в папку cheburashka_only...
[+] Размножаю a5659b27-a0f4-4905-a7f3-c958b42050fa.jpeg в папку cheburashka_only...
[+] Размножаю 2ec3ca3c-e86e-45c0-8fc7-78ed33ab5062.jpeg в папку cheburashka_only...
[+] Размножаю c575fa96-f12f-43f2-b089-b018256e0cc2.jpeg в папку cheburashka_only...
[+] Размножаю 11cdbda6-a3e5-4dfc-9a56-c391e3fd790c.jpeg в папку cheburashka_only...
[+] Размножаю e1a03407-3ca6-448d-9c30-8b1476095a53.jpeg в папку cheburashka_only...
[+] Размножаю cda11ec0-0a9c-407a-99a7-392b86f5ef55.jpeg в папку cheburashka_only...
[+] Размножаю 02f2ee59-d862-412e-95db-5dd208494fcc.jpeg в папку cheburashka_only...
[+] Размножаю 3507671e-00eb-4f05-a4f3-63f48325

In [13]:
import cv2
import numpy as np
import os
import random

# ПУТИ
INPUT_DIR = "/home/alterpath/drones/catkin_ws/src/hack_drones_20-22/augmented_dataset/source/cheburashka_only"
OUTPUT_DIR = "/home/alterpath/drones/catkin_ws/src/hack_drones_20-22/augmented_dataset/classifier_data"

# Целевой размер для обучения (YOLOv8/v11 стандарт)
IMG_SIZE = 640

# Создаем папки
classes = ['background', 'cheburashka_mandarin', 'orange', 'cheburashka_only']
for cls in classes:
    for split in ['train', 'val']:
        os.makedirs(os.path.join(OUTPUT_DIR, split, cls), exist_ok=True)

print(f"Берем файлы в: {INPUT_DIR}")

count = 0
for root, dirs, files in os.walk(INPUT_DIR):
    for file in files:
        if file.lower().endswith((".jpg", ".jpeg")):
            full_path = os.path.join(root, file)
            img = cv2.imread(full_path)
            
            if img is None:
                print(f"[-] Не удалось прочитать: {file}")
                continue

            folder = 'cheburashka_only'

            for i in range(100):
                scale = random.uniform(0.6, 1)
                h, w = img.shape[:2]
                new_w = int(IMG_SIZE * scale)
                new_h = int(h * (new_w / w))
                if new_h > IMG_SIZE:
                    new_h = IMG_SIZE
                    new_w = int(w * (new_h / h))
                
                resized_obj = cv2.resize(img, (new_w, new_h))
                canvas = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype='uint8')
                bg_noise = random.randint(5, 25)
                canvas[:] = bg_noise
                max_x = IMG_SIZE - new_w
                max_y = IMG_SIZE - new_h
                off_x = random.randint(0, max_x)
                off_y = random.randint(0, max_y)
                canvas[off_y:off_y+new_h, off_x:off_x+new_w] = resized_obj

                alpha = random.uniform(0.6, 1.4)
                aug_img = cv2.convertScaleAbs(canvas, alpha=alpha, beta=random.randint(-20, 20))
                
                noise = np.random.randint(0, 15, (IMG_SIZE, IMG_SIZE, 3), dtype='uint8')
                aug_img = cv2.add(aug_img, noise)
                subfolder = 'train' if i < 80 else 'val'
                save_name = f"aug_{i}_{file}"
                save_path = os.path.join(OUTPUT_DIR, subfolder, folder, save_name)
                cv2.imwrite(save_path, aug_img)
            
            count += 1

print(f"Обработано исходников: {count}")
print(f"Сгенерировано изображений: {count * 100}")

Берем файлы в: /home/alterpath/drones/catkin_ws/src/hack_drones_20-22/augmented_dataset/source/cheburashka_only
Обработано исходников: 35
Сгенерировано изображений: 3500


In [23]:
from ultralytics import YOLO
import os


DATA_PATH = "/home/alterpath/drones/catkin_ws/src/hack_drones_20-22/augmented_dataset/classifier_data"

if not os.path.exists(DATA_PATH):
    print(f"ОШИБКА: Папка {DATA_PATH} не найдена!")
    exit()
else:
    train_dir = os.path.join(DATA_PATH, "train")
    classes = os.listdir(train_dir) if os.path.exists(train_dir) else []
    print(f"Папка найдена. Классы для обучения: {classes}")

print("Загружаю базовую модель yolo8n")
model = YOLO('yolov8n-cls.pt')

print("Начинаю обучение")
model.train(
    data=DATA_PATH,
    epochs=40,
    imgsz=640,
    batch=1,
    project='drone_results',
    name='my_classifier'
)

print("Обучение завершено")

Папка найдена. Классы для обучения: ['cheburashka_only', 'orange', 'background', 'cheburashka_mandarin']
Загружаю базовую модель yolo8n
Начинаю обучение
Ultralytics 8.4.24 🚀 Python-3.8.10 torch-2.4.1+cu121 CPU (Intel Core i5-10210U 1.60GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/alterpath/drones/catkin_ws/src/hack_drones_20-22/augmented_dataset/classifier_data, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=40, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max

KeyboardInterrupt: 

In [24]:
import torch
print(f"CUDA доступна: {torch.cuda.is_available()}")

CUDA доступна: False


In [13]:

from ultralytics import YOLO
import cv2

model = YOLO('/home/alterpath/drones/catkin_ws/src/hack_drones_20-22/runs/classify/drone_results/my_classifier9/weights/best.pt')
cap = cv2.VideoCapture(0)

print("Нажми Ctrl+C для выхода. Смотри результат в файле live_view.jpg")

try:
    while cap.isOpened():
        success, frame = cap.read()
        if not success: break

        results = model(frame, verbose=False)
        res = results[0]
        
        # Рисуем текст прямо на кадре (для истории)
        cls_name = res.names[res.probs.top1]
        conf = res.probs.top1conf.item()
        cv2.putText(frame, f"{cls_name} {conf:.2%}", (20, 40), 
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

        # ВМЕСТО cv2.imshow — СОХРАНЯЕМ
        cv2.imwrite("live_view.jpg", frame)
        
        print(f"Вижу: {cls_name} ({conf:.2%})", end='\r')

        # Запускаем классификацию
        results = model(frame, verbose=False)
        res = results[0]
        
        # Получаем вероятность лучшего класса
        conf = res.probs.top1conf.item()
        cls_idx = res.probs.top1
        cls_name = res.names[cls_idx]
        
        # ЖЕСТКИЙ ПОРОГ УВЕРЕННОСТИ
        if conf < 0.90:  # Если уверенность меньше 90%
            final_result = "nothing"
        else:
            final_result = cls_name
                
        print(f"Результат: {final_result} (Уверенность: {conf:.2%})", end='\r')
        # print(f"Результат: {final_result} (Уверенность: {conf:.2%})", end='\r')
except KeyboardInterrupt:
    print("\nСтоп.")
finally:
    cap.release()


Нажми Ctrl+C для выхода. Смотри результат в файле live_view.jpg


[ WARN:0] global ../modules/videoio/src/cap_gstreamer.cpp (935) open OpenCV | GStreamer warning: Cannot query video position: status=0, value=-1, duration=-1


Результат: background (Уверенность: 31.25%))
Стоп.


In [22]:
import pygame
import cv2
import numpy as np
from ultralytics import YOLO
import sys
import time

# 1. ИНИЦИАЛИЗАЦИЯ (сразу проверяем пути)
MODEL_PATH = '/home/alterpath/drones/catkin_ws/src/hack_drones_20-22/runs/classify/drone_results/my_classifier9/weights/best.pt'
CONF_THRESHOLD = 0.90

print("[*] Загрузка YOLO...")
try:
    model = YOLO(MODEL_PATH)
    print("[OK] Модель загружена.")
except Exception as e:
    print(f"[!] Ошибка загрузки модели: {e}")
    sys.exit()

print("[*] Подключение к камере...")
cap = cv2.VideoCapture(0) # Если не работает, попробуй 1 или 2
time.sleep(2.0) # Даем время драйверу проснуться

# 2. НАСТРОЙКА PYGAME
pygame.init()
screen = pygame.display.set_mode((800, 600))
pygame.display.set_caption("YOLO LIVE CHECKER")
font = pygame.font.SysFont("Arial", 24)
clock = pygame.time.Clock()

print("[*] Старт цикла. Нажми ESC для выхода.")

# 3. ОСНОВНОЙ ЦИКЛ
running = True
while running:
    # Проверка очереди событий Pygame (чтобы окно не зависло)
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False
        if event.type == pygame.KEYDOWN:
            if event.key == pygame.K_ESCAPE:
                running = False

    # Читаем кадр
    success, frame = cap.read()
    
    if not success:
        # Если кадра нет, рисуем заглушку и ждем, не выходя из программы
        screen.fill((100, 0, 0)) # Красный фон — сигнал ошибки
        error_txt = font.render("WAITING FOR CAMERA...", True, (255, 255, 255))
        screen.blit(error_txt, (300, 280))
        pygame.display.flip()
        time.sleep(0.5)
        continue

    # КЛАССИФИКАЦИЯ
    results = model(frame, verbose=False)
    res = results[0]
    conf = res.probs.top1conf.item()
    cls_name = res.names[res.probs.top1]
    
    # Решение по порогу
    final_result = cls_name if conf >= CONF_THRESHOLD else "nothing"

    # ВИЗУАЛИЗАЦИЯ (Конвертация кадра)
    # BGR -> RGB
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    # Поворот для Pygame
    rgb_frame = np.rot90(rgb_frame)
    img_surface = pygame.surfarray.make_surface(rgb_frame)
    img_surface = pygame.transform.flip(img_surface, True, False)
    img_surface = pygame.transform.scale(img_surface, (640, 480))

    # Отрисовка на экран
    screen.fill((30, 30, 30))
    screen.blit(img_surface, (80, 20))

    # Получаем топ-2 результата
    top2_indices = res.probs.top5[:2] # Берем первые два индекса
    
    screen.fill((30, 30, 30))
    screen.blit(img_surface, (80, 20))

    # Вывод подробной отладки
    for i, idx in enumerate(top2_indices):
        name = res.names[int(idx)]
        probability = res.probs.data[int(idx)].item()
        
        # Если это наш главный результат и он выше порога
        color = (0, 255, 0) if (i == 0 and probability >= CONF_THRESHOLD) else (200, 200, 200)
        
        debug_txt = font.render(f"{i+1}. {name}: {probability:.2%}", True, color)
        screen.blit(debug_txt, (80, 520 + (i * 30)))

    pygame.display.flip()
    clock.tick(30) # 30 кадров в секунду

# 4. ЗАВЕРШЕНИЕ (только по Esc или закрытию окна)
cap.release()
print("[*] Программа остановлена.")

# Финальное окно, чтобы не закрылось сразу после ошибки
waiting = True
while waiting:
    screen.fill((0, 0, 0))
    finish_txt = font.render("FINISHED. PRESS ANY KEY TO EXIT.", True, (255, 255, 0))
    screen.blit(finish_txt, (200, 280))
    pygame.display.flip()
    for event in pygame.event.get():
        if event.type == pygame.QUIT or event.type == pygame.KEYDOWN:
            waiting = False

pygame.quit()

[*] Загрузка YOLO...
[OK] Модель загружена.
[*] Подключение к камере...


[ WARN:0] global ../modules/videoio/src/cap_gstreamer.cpp (935) open OpenCV | GStreamer warning: Cannot query video position: status=0, value=-1, duration=-1


[*] Старт цикла. Нажми ESC для выхода.
[*] Программа остановлена.


In [69]:
import numpy as np
import cv2
import torch
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator

# 1. Настройки модели (нужно скачать веса 'sam_vit_h.pth')
sam_checkpoint = "sam_vit_h_4b8939.pth"
model_type = "vit_h"
device = "cuda" if torch.cuda.is_available() else "cpu"

sam = sam_model_registry[model_type](checkpoint=sam_checkpoint)
sam.to(device=device)

# Генератор масок
mask_generator = SamAutomaticMaskGenerator(sam)

# 2. Твое изображение пола
img_path = "/home/alterpath/drones/catkin_ws/src/hack_drones_20-22/augmented_dataset/source/0.jpeg"
image = cv2.imread(img_path)
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# 3. ГЕНЕРАЦИЯ МАСОК (Автоматически найдет все ArUco)
print("[*] Сегментирую пол... это может занять время.")
masks = mask_generator.generate(image)
print(f"[*] Найдено объектов: {len(masks)}")

# 4. Сохранение масок в формат для обучения YOLO-Seg
for i, mask in enumerate(masks):
    binary_mask = (mask['segmentation'] * 255).astype(np.uint8)
    # Сохраняй маски в отдельные файлы, например
    # mask_for_target_0.png
    cv2.imwrite(f"mask_{i}.png", binary_mask)

print("[ГОТОВО] Маски сохранены.")

FileNotFoundError: [Errno 2] No such file or directory: 'sam_vit_h_4b8939.pth'

In [66]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("fareselmenshawii/human-dataset")

print("Path to dataset files:", path)

/home/alterpath/.local/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  5%|██                                     | 281M/5.33G [01:11<22:00, 4.11MB/s]


KeyboardInterrupt: 

In [27]:

from ultralytics import YOLO

def start_training():
    model = YOLO('yolov8n.pt') 

    model.train(
        data='/home/alterpath/drones/catkin_ws/src/hack_drones_20-22/augmented_dataset/detect_data/data.yaml',
        epochs=40,
        imgsz=410,
        batch=16,
        device='cpu',
        project='cheb_project',
        name='v1_augmented',
        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4,
        degrees=15.0,
        translate=0.1,
        scale=0.5,
        shear=2.0,
        perspective=0.001,
        flipud=0.0,
        fliplr=0.5,
        mosaic=1.0,
        mixup=0.1
    )

if __name__ == '__main__':
    start_training()

Ultralytics 8.4.24 🚀 Python-3.8.10 torch-2.4.1+cu121 CPU (Intel Core i5-10210U 1.60GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/alterpath/drones/catkin_ws/src/hack_drones_20-22/augmented_dataset/detect_data/data.yaml, degrees=15.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=40, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=410, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=v1_augmented2, nbs=64, nms=False, opset=None